# Lección 9.6 — Hiperparámetros de la red: el tablero de perillas

> **Módulo 9 · Redes neuronales desde la intuición** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 110–130 min (lección con ejercicios intercalados ≈ 80 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 3.4 (hiperparámetros del entrenamiento), Lección 3.5 (mini-batch), Lección 3.7 (curvas de aprendizaje), Lección 7.2 (*early stopping*), Lección 7.3 (búsqueda de hiperparámetros), Lección 9.5 (optimizadores)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué controla cada perilla de una red: **tasa de aprendizaje, *scheduler*, tamaño de lote, épocas, número de capas y de neuronas**.
- Encontrar una buena tasa de aprendizaje con un **barrido** y con el ***LR range test***.
- Reconocer la forma de los *schedulers* más usados (escalones, coseno, *warmup* + coseno, *one-cycle*) y saber cuándo ayudan.
- Decidir **cuántas épocas** entrenar con *early stopping* y en qué **orden** ajustar las perillas.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.
>
> ⏳ Esta lección entrena bastantes redes pequeñas. En una computadora normal, cada celda tarda entre unos segundos y medio minuto.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas y los datos: 10,000 dígitos de **MNIST** para entrenar y 2,000 para validar.

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets

np.random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (8, 4.5)

mnist = datasets.MNIST("data", train=True, download=True)
X_todo = mnist.data.float().div(255).view(-1, 784)     # píxeles 0..1, aplanados
y_todo = mnist.targets
X_tr, y_tr = X_todo[:10_000], y_todo[:10_000]           # entrenamiento
X_va, y_va = X_todo[50_000:52_000], y_todo[50_000:52_000]   # validación
criterio = nn.CrossEntropyLoss()
print("✅ Todo listo · dispositivo:", device, "· entrenamiento:", len(X_tr), "· validación:", len(X_va))

Usaremos durante toda la lección **una sola función** que entrena una red con las perillas que le pidas y devuelve su historia. No necesitas leerla en detalle: es el bucle de la Lección 9.4 con mini-lotes.

In [ ]:
def crear_red(capas=1, neuronas=128):
    bloques, entrada = [], 784
    for _ in range(capas):
        bloques += [nn.Linear(entrada, neuronas), nn.ReLU()]
        entrada = neuronas
    return nn.Sequential(*bloques, nn.Linear(entrada, 10))

def entrenar(lr=1e-3, lote=64, epocas=5, capas=1, neuronas=128, scheduler=None,
             X_tr=X_tr, y_tr=y_tr, X_va=X_va, y_va=y_va):
    """Entrena con Adam y devuelve un diccionario con la historia por época."""
    torch.manual_seed(0)
    red = crear_red(capas, neuronas)
    opt = torch.optim.Adam(red.parameters(), lr=lr)
    pasos_totales = epocas * math.ceil(len(X_tr) / lote)
    sched = scheduler(opt, pasos_totales) if scheduler else None
    h = {"perdida_tr": [], "perdida_va": [], "exactitud_va": []}
    inicio = time.time()
    for epoca in range(epocas):
        orden, total = torch.randperm(len(X_tr)), 0
        for i in range(0, len(X_tr), lote):
            idx = orden[i:i + lote]
            opt.zero_grad()
            perdida = criterio(red(X_tr[idx]), y_tr[idx])
            perdida.backward()
            opt.step()
            if sched: sched.step()          # el scheduler ajusta la tasa en CADA paso
            total += perdida.item() * len(idx)
        with torch.no_grad():
            salida = red(X_va)
            h["perdida_va"].append(criterio(salida, y_va).item())
            h["exactitud_va"].append((salida.argmax(1) == y_va).float().mean().item())
        h["perdida_tr"].append(total / len(X_tr))
    h["segundos"] = time.time() - inicio
    h["parametros"] = sum(p.numel() for p in red.parameters())
    return h

---
## 1. 📖 La intuición: hornear pan

Una red neuronal se parece a **hornear pan** con un horno nuevo:

| Perilla de la red | En la cocina | Qué pasa si te equivocas |
|---|---|---|
| **Tasa de aprendizaje** | Temperatura del horno | Muy alta: se quema por fuera y queda crudo (diverge). Muy baja: nunca se cuece |
| ***Scheduler*** | Programa del horno: precalentar, hornear fuerte y bajar al final | Sin programa: o se dora mal o tarda demasiado |
| **Épocas** | Tiempo en el horno | Poco: crudo (subajuste). Demasiado: seco y duro (sobreajuste) |
| **Tamaño de lote** | Cuántos panes metes por tanda | Tandas grandes: más rápido por pan, pero cada tanda "aprende" menos del horno |
| **Capas y neuronas** | Tamaño del horno | Pequeño: no cabe el pan (subajuste). Enorme: desperdicias y es fácil quemarlo |

Un buen panadero no gira todas las perillas a la vez: **primero clava la temperatura**, porque si está mal, nada más importa.

### ✋ Ejercicio 1 — Calentamiento
Tu pan sale **crudo**. Antes de comprar un horno más grande (más neuronas), ¿qué dos perillas revisarías primero y por qué?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

La **temperatura** (tasa de aprendizaje) y el **tiempo** (épocas). Son las causas más comunes y las más baratas de cambiar. Comprar un horno más grande cuando el problema es que no lo dejaste suficiente tiempo no sirve de nada. En redes pasa igual: antes de agrandar la red, comprueba que la tasa es razonable y que entrenaste lo suficiente.
</details>

---
## 2. 📖 La perilla número 1: la tasa de aprendizaje

Ya la conoces (Lecciones 3.4 y 9.5): el tamaño de cada paso. En redes es, con diferencia, **la perilla más importante**. Una tasa mala arruina cualquier otra elección; una buena tasa hace que casi todo lo demás "funcione bien".

Como sus buenos valores pueden estar en cualquier escala, se prueba en **potencias de 10**: 0.00001, 0.0001, 0.001, 0.01, 0.1.

### 🔮 Ejercicio 2 — El barrido
Entrenamos 5 épocas con Adam y las 5 tasas. Antes de ejecutar:
1. ¿Cuál crees que ganará en validación?
2. ¿Habrá alguna que quede **peor que las tasas pequeñas**, aunque sus pasos sean más grandes?

✍️ *Mis predicciones:*

In [ ]:
for lr in [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]:          # ✏️
    h = entrenar(lr=lr)
    plt.plot(range(1, 6), h["perdida_tr"], "o-", label=f"lr={lr:g} · validación {h['exactitud_va'][-1]:.1%}")
    print(f"lr={lr:<7g} pérdida final {h['perdida_tr'][-1]:.3f} · exactitud validación {h['exactitud_va'][-1]:.1%}")
plt.yscale("log"); plt.xlabel("época"); plt.ylabel("pérdida de entrenamiento (log)")
plt.title("Barrido de la tasa de aprendizaje"); plt.legend(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. Gana **`lr=0.01`** (≈94 %), seguida muy de cerca por `0.001` (≈93 %). Ojo: el valor de partida de Adam (`1e-3`) no era el mejor, pero estaba cerca.
2. **Sí: `lr=0.1`** queda en ≈64 %, ¡peor incluso que la tasa diminuta `0.00001` (≈69 %)! Sus pasos son tan grandes que rebota por el terreno sin asentarse. Las tasas pequeñas, en cambio, van en la dirección correcta pero no les alcanzan 5 épocas (`0.0001` → ≈88 %).

La curva típica de exactitud contra tasa es una **montaña**: sube, llega a una meseta buena (aquí entre 0.001 y 0.01) y se desploma cuando la tasa es demasiado grande.
</details>

---
## 3. 📖 El *LR range test*: encontrar la tasa en una sola pasada

Un barrido entrena varias redes completas. Hay un truco más barato, el ***LR range test*** (prueba de rango de tasas): entrenas **una sola vez**, durante unos pocos cientos de pasos, **subiendo la tasa** en cada paso, desde una diminuta hasta una enorme, y anotas la pérdida.

```
pérdida
  │ ‾‾‾‾‾‾‾\                         /
  │         \                       /   ← aquí explota: tasa demasiado grande
  │          \_______         ____/
  │   casi no       \_______/
  │   se mueve        ↑ baja rápido
  └───────────────────────────────────── tasa (escala log)
```

**Regla práctica:** elige una tasa **unas 10 veces menor** que la tasa donde la pérdida llega a su mínimo (ahí ya está a punto de explotar).

In [ ]:
def buscar_lr(X, y, lr_min=1e-6, lr_max=1.0, pasos=200, lote=64):
    torch.manual_seed(0)
    red = crear_red()
    opt = torch.optim.Adam(red.parameters(), lr=lr_min)
    factor = (lr_max / lr_min) ** (1 / pasos)           # la tasa se multiplica por esto en cada paso
    tasas, perdidas, suave = [], [], None
    for paso in range(pasos):
        idx = torch.randint(0, len(X), (lote,))
        opt.zero_grad()
        perdida = criterio(red(X[idx]), y[idx])
        perdida.backward()
        opt.step()
        suave = perdida.item() if suave is None else 0.9 * suave + 0.1 * perdida.item()   # suavizamos el ruido
        tasas.append(opt.param_groups[0]["lr"]); perdidas.append(suave)
        if suave > 4 * min(perdidas):                   # ya explotó: no hace falta seguir
            break
        opt.param_groups[0]["lr"] *= factor
    return np.array(tasas), np.array(perdidas)

tasas, perdidas = buscar_lr(X_tr, y_tr)
mejor = tasas[perdidas.argmin()]
plt.plot(tasas, perdidas); plt.xscale("log"); plt.axvline(mejor, color="r", ls="--")
plt.axvline(mejor / 10, color="g", ls="--")
plt.xlabel("tasa de aprendizaje (log)"); plt.ylabel("pérdida (suavizada)")
plt.title("LR range test · rojo: mínimo · verde: 10 veces menos"); plt.show()
print(f"La pérdida llega a su mínimo con lr ≈ {mejor:.3g} → tasa sugerida ≈ {mejor / 10:.3g}")

### ✋ Ejercicio 3 — Lee la gráfica
1. ¿En qué rango de tasas la pérdida **casi no se mueve**?
2. ¿La tasa sugerida coincide con lo que encontraste en el barrido del Ejercicio 2?
3. ¿Cuántas redes entrenaste para obtenerla, comparado con el barrido?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Por debajo de ≈0.0001 la pérdida casi no se mueve (se queda en ≈2.3, adivinar al azar): los pasos son demasiado pequeños para notarse en tan pocos pasos. Empieza a bajar con fuerza entre ≈0.001 y ≈0.01, toca fondo hacia ≈0.015–0.05 y explota más allá de ≈0.1.
2. **Sí, a grandes rasgos:** sugiere ≈0.0015, dentro de la "meseta buena" del barrido (0.001–0.01). Es una sugerencia **prudente**: el barrido encontró que 0.01 era un poco mejor. El *LR range test* te da la **zona** segura, no el número exacto; después puedes afinar con 2–3 entrenamientos alrededor.
3. **Una sola** red, y ni siquiera una época completa (menos de 200 pasos), contra 5 redes de 5 épocas del barrido.
</details>

---
## 4. 📖 *Schedulers*: cambiar la tasa durante el entrenamiento

¿Por qué usar **la misma** tasa todo el tiempo? Al principio estás lejos del valle y conviene dar pasos grandes; al final estás cerca del fondo y conviene dar pasos pequeños para no rebotar. Un ***scheduler*** (programador de la tasa) cambia la tasa **automáticamente** a lo largo del entrenamiento.

| *Scheduler* | Forma | En PyTorch |
|---|---|---|
| **Escalones** (*step decay*) | Constante, y cada cierto número de pasos se divide (p. ej. entre 10) | `StepLR` |
| **Coseno** (*cosine annealing*) | Baja suavemente, como media ola, hasta casi cero | `CosineAnnealingLR` |
| ***Warmup* + coseno** | Sube poco a poco al principio ("calentar") y luego baja en coseno | `LambdaLR` con una fórmula |
| ***One-cycle*** | Sube hasta un máximo y luego baja muy por debajo del inicio | `OneCycleLR` |

> 💡 **¿Por qué calentar (*warmup*)?** Al inicio los pesos son aleatorios y los gradientes pueden ser enormes y poco confiables. Unos cuantos pasos con tasa pequeña evitan un primer "golpe" que desestabilice la red. Es imprescindible en Transformers (Módulo 12).

Veamos sus formas sobre 100 pasos de entrenamiento.

In [ ]:
def warmup_coseno(opt, pasos_totales, fraccion_warmup=0.1):
    calentar = max(1, int(fraccion_warmup * pasos_totales))
    def factor(paso):
        if paso < calentar:
            return (paso + 1) / calentar                              # sube en línea recta
        avance = (paso - calentar) / max(1, pasos_totales - calentar)
        return 0.5 * (1 + math.cos(math.pi * avance))                 # baja en coseno
    return torch.optim.lr_scheduler.LambdaLR(opt, factor)

schedulers = {
    "Escalones (StepLR)": lambda opt, n: torch.optim.lr_scheduler.StepLR(opt, step_size=n // 3, gamma=0.1),
    "Coseno":             lambda opt, n: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n),
    "Warmup + coseno":    warmup_coseno,
    "One-cycle":          lambda opt, n: torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=opt.param_groups[0]["lr"], total_steps=n),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
for ax, (nombre, crear) in zip(axes, schedulers.items()):
    opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=1.0)   # una "red" de mentira
    sched, tasas_vistas = crear(opt, 100), []
    for paso in range(100):
        tasas_vistas.append(opt.param_groups[0]["lr"]); opt.step(); sched.step()
    ax.plot(tasas_vistas); ax.set_title(nombre); ax.set_xlabel("paso")
axes[0].set_ylabel("tasa (relativa a la máxima)"); plt.tight_layout(); plt.show()

### 🔮 Ejercicio 4 — ¿Cuál es cuál?
Sin mirar los títulos de la gráfica:
1. ¿Cuál de los cuatro *schedulers* **empieza** con una tasa pequeña?
2. ¿En cuál la tasa cambia de golpe?
3. En el *one-cycle*, ¿la tasa termina por encima o por debajo de donde empezó?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Warmup + coseno** y **one-cycle**: ambos empiezan bajo y suben. El *warmup* sube en línea recta durante el 10 % de los pasos; el *one-cycle* sube en curva durante ~30 % (empieza en la máxima / 25).
2. **Escalones**: se queda plana y cae de golpe (×0.1) en los pasos 33 y 66.
3. **Muy por debajo**: el *one-cycle* termina con una tasa diminuta (cientos de miles de veces menor que la máxima). Así la red "se asienta" en el fondo del valle.
</details>

### 🔮 Ejercicio 5 — ¿Sirven de algo?
Entrenamos 8 épocas con una tasa **alta** (`lr=0.01`, la mejor del barrido) con cada *scheduler*, y una vez sin *scheduler*. Antes de ejecutar: ¿cuántos puntos de exactitud crees que gana el mejor *scheduler* frente a la tasa constante: ninguno, ~1–2 o ~10?

✍️ *Mi apuesta:*

In [ ]:
opciones = {"Sin scheduler (constante)": None} | {k: v for k, v in schedulers.items() if k != "Escalones (StepLR)"}
for nombre, sched in opciones.items():
    h = entrenar(lr=0.01, epocas=8, scheduler=sched)
    plt.plot(range(1, 9), h["perdida_va"], "o-", label=nombre)
    print(f"{nombre:<26} pérdida entrenamiento {h['perdida_tr'][-1]:.3f} · pérdida validación {h['perdida_va'][-1]:.3f} · exactitud validación {h['exactitud_va'][-1]:.1%}")
plt.xlabel("época"); plt.ylabel("pérdida de validación"); plt.legend(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

**~1–2 puntos:** sin *scheduler*, ≈94.5 %; con coseno, *warmup* + coseno o *one-cycle*, ≈96 % los tres. Fíjate también en la **pérdida de validación**: baja de ≈0.27 a ≈0.15, casi la mitad.

¿Por qué? Con la tasa alta constante, la red sigue "rebotando" alrededor del fondo del valle hasta el final. Los *schedulers* terminan con una tasa casi cero, así que la red **se asienta**. Entre los tres *schedulers* casi no hay diferencia: lo importante es **bajar la tasa al final**, más que la forma exacta.
</details>

---
## 5. 📖 Tamaño de lote (*batch size*)

En la Lección 3.5 viste que cada paso usa un **mini-lote** de ejemplos. Su tamaño cambia tres cosas a la vez:

| Lote pequeño (8–32) | Lote grande (512–4096) |
|---|---|
| Muchos pasos por época → aprende mucho por época | Pocos pasos por época |
| Gradiente **ruidoso** (a veces ayuda a generalizar) | Gradiente estable |
| Lento en tiempo de reloj: no aprovecha el paralelismo | Rápido por época (sobre todo en GPU) |
| Poca memoria | Mucha memoria |

### 🔮 Ejercicio 6
Entrenamos **5 épocas** con la **misma tasa** (`lr=0.001`) y lotes de 16, 64, 256 y 1024. ¿Qué lote crees que dará la mejor exactitud? ¿Y cuál será el más rápido?

✍️ *Mis predicciones:*

In [ ]:
for lote in [16, 64, 256, 1024]:          # ✏️
    h = entrenar(lote=lote)
    pasos = 5 * math.ceil(len(X_tr) / lote)
    print(f"lote={lote:<5} pasos totales={pasos:<5} tiempo={h['segundos']:5.1f} s · "
          f"pérdida {h['perdida_tr'][-1]:.3f} · exactitud validación {h['exactitud_va'][-1]:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Mejor exactitud: el lote más pequeño (16)**, ≈95 %, y va empeorando con lotes más grandes hasta ≈88 % con 1024.
- **Más rápido: el lote 1024**, unas 20 veces más rápido que el de 16 en nuestra prueba (en GPU la diferencia sería aún mayor).

Mira la columna "pasos totales": con lote 16 la red dio 3,125 pasos; con 1024, solo 50. Con la misma tasa y las mismas épocas, el lote grande **aprendió mucho menos**. No es que los lotes grandes sean malos: es que comparamos con "las mismas épocas", no con "la misma cantidad de aprendizaje".
</details>

### ✏️ Ejercicio 7 — Lote grande, tasa más grande
Una regla muy usada: **si multiplicas el lote por k, multiplica la tasa por k** (regla lineal; algunos prefieren por √k), porque con lotes más grandes el gradiente es más fiable y das menos pasos. Comprueba si con lote 1024 recuperas exactitud subiendo la tasa.

In [ ]:
LOTE = 1024
for lr in [0.001, 0.004, 0.016]:          # ✏️ ×1, ×4 (≈√16), ×16 (lineal, porque 1024 = 64 × 16)
    h = entrenar(lote=LOTE, lr=lr)
    print(f"lote={LOTE} lr={lr:<6} → pérdida {h['perdida_tr'][-1]:.3f} · exactitud validación {h['exactitud_va'][-1]:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Sí, en buena parte: `0.001` → ≈88 %, `0.004` → ≈91 %, `0.016` → ≈93 %, casi como el lote 64 con su tasa original (≈93 %), pero en una fracción del tiempo. Con lotes grandes, el gradiente es más fiable y aguanta pasos más largos.

La regla no es exacta ni funciona sin límite (con lotes enormes deja de cumplirse y suele hacer falta *warmup*), pero es un buen punto de partida: **cambiar el lote sin reajustar la tasa es una comparación injusta.**
</details>

---
## 6. 📖 Épocas y *early stopping*

Una **época** es una pasada completa por los datos de entrenamiento. ¿Cuántas? La respuesta honesta: **las que diga la curva de validación** (Lecciones 3.7 y 7.2). Se entrena "de más" y se guarda el modelo de la época en que la **pérdida de validación** fue mínima. Eso es el ***early stopping*** (parada temprana).

Para ver el problema con claridad, entrenamos una red **grande** (2 capas de 512) con **pocos datos** (solo 1,000 imágenes) durante 40 épocas.

In [ ]:
h = entrenar(epocas=40, capas=2, neuronas=512, X_tr=X_tr[:1000], y_tr=y_tr[:1000])
mejor_epoca = int(np.argmin(h["perdida_va"])) + 1

plt.plot(range(1, 41), h["perdida_tr"], label="entrenamiento")
plt.plot(range(1, 41), h["perdida_va"], label="validación")
plt.axvline(mejor_epoca, color="g", ls="--", label=f"mínimo de validación (época {mejor_epoca})")
plt.yscale("log"); plt.xlabel("época"); plt.ylabel("pérdida (log)"); plt.legend(); plt.show()
print(f"Mejor época: {mejor_epoca} · pérdida validación {min(h['perdida_va']):.3f} · exactitud {h['exactitud_va'][mejor_epoca - 1]:.1%}")
print(f"Época 40   : pérdida validación {h['perdida_va'][-1]:.3f} · exactitud {h['exactitud_va'][-1]:.1%}")

### 🔮 Ejercicio 8 — Lee la historia
1. ¿Qué le pasa a la pérdida de **entrenamiento**? ¿Y a la de **validación** después de la mejor época?
2. Mira la exactitud de validación al final. ¿Empeoró tanto como la pérdida? ¿Cómo puede ser?
3. Con *paciencia* = 5 (parar si en 5 épocas seguidas no mejora la validación), ¿en qué época se habría detenido?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. La de **entrenamiento** baja hasta casi cero: la red memoriza las 1,000 imágenes. La de **validación** toca fondo en la **época 7** (≈0.42) y después **sube** hasta ≈0.55 en la época 40: sobreajuste de manual (Lección 4.4).
2. **No:** la exactitud de validación casi no cambia (≈87.5 % en la época 7, ≈88 % en la 40). La red sigue eligiendo la misma clase en casi todas las imágenes, pero se vuelve **demasiado segura de sí misma**: cuando se equivoca, se equivoca con probabilidades altísimas, y la entropía cruzada la castiga cada vez más. Por eso la pérdida de validación es mejor alarma que la exactitud.
3. En la **época 12**: la mejor fue la 7 y en las 5 siguientes (8 a 12) no hubo ninguna mejor, así que se para ahí y **se recupera el modelo de la época 7**. Ahorras 28 épocas de cómputo y te quedas con el modelo más sano.
</details>

---
## 7. 📖 Capas y neuronas: el tamaño del horno

El número de **capas** (profundidad) y de **neuronas por capa** (ancho) fija la **capacidad** de la red: cuántos patrones distintos puede representar (Lección 9.2). Más capacidad = puede aprender más, pero también memorizar más y tardar más.

### 🔮 Ejercicio 9
Comparamos redes de 1 y 3 capas ocultas con 16, 64 y 256 neuronas por capa (5 épocas). ¿Qué importa más aquí: **agregar capas** o **agregar neuronas**?

✍️ *Mi predicción:*

In [ ]:
for capas in [1, 3]:                      # ✏️ prueba 2
    for neuronas in [16, 64, 256]:
        h = entrenar(capas=capas, neuronas=neuronas)
        print(f"capas={capas} neuronas={neuronas:<4} parámetros={h['parametros']:>8,} · "
              f"pérdida {h['perdida_tr'][-1]:.3f} · exactitud validación {h['exactitud_va'][-1]:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

**Agregar neuronas** importa más aquí. Con 1 capa, pasar de 16 a 256 neuronas sube de ≈91 % a ≈93 %. Agregar capas **con pocas neuronas** incluso empeora (3 capas de 16: ≈89 %): cada capa estrecha es un "embudo" que pierde información y la red es más difícil de entrenar en 5 épocas. Con 256 neuronas, en cambio, 3 capas sí ganan (≈95 %, la mejor), aunque con más parámetros (≈335 mil contra ≈204 mil).

Lección: la profundidad ayuda cuando cada capa tiene **suficiente ancho**. Para datos tabulares o MNIST, 1–3 capas razonablemente anchas suelen bastar; las redes muy profundas brillan con imágenes, texto y audio (Módulos 10–12).
</details>

---
## 8. 📖 ¿En qué orden giro las perillas?

No hay tiempo para probar todas las combinaciones (Lección 7.3). Este es el orden que funciona en la práctica:

1. **Elige un punto de partida conocido:** Adam, `lr=1e-3`, lote 32–128, una red "razonable" (p. ej. 1–2 capas de 128–256), un número de épocas generoso.
2. **Ajusta la tasa de aprendizaje** (barrido en potencias de 10 o *LR range test*). Es la que más impacto tiene.
3. **Fija las épocas con *early stopping*** mirando la validación.
4. **Ajusta la capacidad** (neuronas y capas): si la red **subajusta** (entrenamiento y validación malos), agrándala; si **sobreajusta**, regularízala (Lección 9.8) antes de achicarla.
5. **Añade un *scheduler*** (coseno o *one-cycle*): suele dar un último empujón casi gratis.
6. **Tamaño de lote:** el más grande que quepa en memoria y no empeore la validación; si lo cambias, **reajusta la tasa**.
7. Todo lo demás (betas, eps, momentum…) casi nunca lo tocarás.

### 📋 Chuleta de hiperparámetros de la red

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| Tasa de aprendizaje (`lr`) | Tamaño de los pasos | Más rápido hasta oscilar o divergir (NaN) | Lento; se queda corto con pocas épocas | Adam `1e-3`; luego *LR range test* |
| *Scheduler* | Cómo cambia la tasa con el tiempo | — | — | Coseno o *one-cycle*; *warmup* (≈5–10 % de los pasos) en Transformers |
| Tamaño de lote | Ejemplos por paso | Más rápido por época, gradiente estable, más memoria; menos pasos | Gradiente ruidoso, más pasos, más lento | 32–128 |
| Épocas | Pasadas por los datos | Riesgo de **sobreajuste** | Riesgo de **subajuste** | Generosas + *early stopping* (paciencia 5–10) |
| Neuronas por capa | Ancho, capacidad | Más capacidad, más lento, más riesgo de sobreajuste | Riesgo de subajuste | 64–512 (potencias de 2 por costumbre) |
| Número de capas | Profundidad, capacidad | Más capacidad, más difícil de entrenar (Lección 9.7) | Más simple | 1–3 capas ocultas para datos tabulares/MNIST |

### ✋ Ejercicio 10
Tu compañero empieza un proyecto nuevo y su primer paso es probar 5 tamaños de lote × 4 números de capas × 3 *schedulers* = 60 entrenamientos, con `lr=1e-3` fijo. ¿Qué le recomendarías hacer antes?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Que **primero ajuste la tasa de aprendizaje** (un barrido de 4–5 valores o un *LR range test*: unos minutos). Si `1e-3` resulta ser una mala tasa para su red, sus 60 entrenamientos estarán comparando versiones todas igual de "mal horneadas", y las conclusiones no valdrán. Además, el lote y la tasa interactúan (Ejercicio 7): no tiene sentido variar el lote sin reajustar la tasa.
</details>

---
## 9. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| La pérdida no baja o lo hace lentísimo | Tasa demasiado baja | Sube `lr` ×10; *LR range test* |
| La pérdida oscila, sube o da `NaN` | Tasa demasiado alta | Baja `lr` ×10; añade *warmup*; recorte de gradientes (Lección 9.7) |
| Cambias el lote y "todo empeora" | La tasa ya no corresponde al lote | Reajusta `lr` (regla lineal o √k) |
| La validación mejora y luego empeora | Demasiadas épocas → sobreajuste | *Early stopping*; regularización (Lección 9.8) |
| Entrenamiento y validación igual de malos | Red demasiado pequeña o pocas épocas → subajuste | Más épocas, más neuronas |
| Tomas decisiones mirando el conjunto de **prueba** | Fuga de información (Lección 4.6) | Decide con validación; prueba una sola vez al final |
| "Encontré el lote perfecto: 37" | Sobreinterpretar diferencias de ruido | Usa valores redondos; repite con otra semilla antes de creer diferencias de décimas |

### 🔮 Ejercicio 11
Un entrenamiento iba bien. Cambiaste el lote de 32 a 512 para que fuera más rápido, dejaste todo lo demás igual, y ahora la exactitud de validación es 4 puntos peor tras el mismo número de épocas. ¿Cuál es la explicación más probable y qué harías?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con un lote 16 veces mayor, la red da **16 veces menos pasos** por época con la misma tasa: simplemente aprendió menos (es lo que viste en el Ejercicio 6). Prueba a subir la tasa (×4 a ×16, Ejercicio 7), o a entrenar más épocas. No es que los lotes grandes "sean malos".
</details>

---
## 10. 🧠 Resumen en 5 puntos
1. La **tasa de aprendizaje** es la perilla más importante: búscala primero, en potencias de 10 o con un ***LR range test*** (≈10 veces menos que donde la pérdida toca fondo).
2. Un ***scheduler*** (coseno, *warmup* + coseno, *one-cycle*) permite pasos grandes al principio y pequeños al final; con una tasa alta, suele dar una mejora casi gratis.
3. El **tamaño de lote** cambia el número de pasos, el ruido y la velocidad; si lo cambias, **reajusta la tasa**.
4. Las **épocas** las decide la curva de validación: entrena de más y quédate con la mejor (***early stopping***).
5. **Capas y neuronas** fijan la capacidad: agranda si subajusta, regulariza si sobreajusta. Y ajusta en orden: tasa → épocas → capacidad → *scheduler* → lote.

---
## 11. 🏋️ Ejercicios finales

Ya practicaste cada perilla por separado. Ahora toca **combinarlas** (≈35 min).

### 11.1 Repaso rápido (opción múltiple)

**1.** Si solo pudieras ajustar **un** hiperparámetro de tu red, ¿cuál elegirías?
- a) El número de capas · b) La tasa de aprendizaje · c) El tamaño de lote · d) `eps` de Adam

**2.** En un *LR range test*, la pérdida llega a su mínimo con `lr ≈ 0.05`. ¿Qué tasa eliges?
- a) 0.5 · b) 0.05 · c) 0.005 · d) 0.00005

**3.** ¿Para qué sirve el *warmup*?
- a) Para terminar el entrenamiento con pasos pequeños
- b) Para evitar pasos grandes al inicio, cuando los pesos aleatorios dan gradientes poco confiables
- c) Para precalentar la GPU
- d) Para aumentar el tamaño de lote

**4.** Duplicas el tamaño de lote sin tocar nada más. ¿Qué pasa con el número de pasos por época?
- a) Se duplica · b) Se queda igual · c) Se reduce a la mitad · d) Depende del optimizador

**5.** La pérdida de validación es mínima en la época 12 y sube después; entrenaste 50. ¿Qué modelo usas?
- a) El de la época 50 · b) El de la época 12 · c) El de la época 1 · d) Un promedio de todos

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Es la perilla con más impacto, de lejos.
2. **c)** ≈10 veces menos que el mínimo, que ya está al borde de explotar.
3. **b)** Calentar suavemente evita que un primer paso enorme desestabilice la red.
4. **c)** Mismos datos repartidos en lotes el doble de grandes → la mitad de pasos.
5. **b)** Eso es *early stopping*: te quedas con el modelo del mínimo de validación.
</details>

### 11.2 Preguntas para pensar (sin ejecutar)

**A.** En el Ejercicio 8 la pérdida de validación subía mucho, pero la exactitud casi no bajaba. Si tu modelo va a dar **probabilidades** a un médico ("70 % de probabilidad de enfermedad"), ¿qué curva vigilarías para el *early stopping* y por qué?

**B.** Una empresa entrena un modelo enorme una sola vez, con un presupuesto fijo de 10,000 pasos. ¿Por qué le conviene un *scheduler* que termine con una tasa casi cero (coseno, *one-cycle*) en lugar de una tasa constante?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** La **pérdida de validación**. La exactitud solo mira si la clase con más probabilidad es la correcta; la pérdida (entropía cruzada) castiga además que el modelo esté **demasiado seguro** cuando se equivoca. Una red sobreajustada sigue acertando más o menos lo mismo, pero sus probabilidades se vuelven exageradas (99 % cuando debería decir 70 %), y eso para un médico es engañoso.

**B.** Con presupuesto fijo, quieres avanzar rápido al principio (tasa alta) y llegar al final **asentado** en el fondo del valle (tasa casi cero), en lugar de seguir rebotando alrededor del mínimo cuando se acaban los pasos. Un *scheduler* hace las dos cosas en un solo entrenamiento. Por eso casi todos los LLMs se entrenan con *warmup* + coseno (Módulo 13).
</details>

### 11.3 🧩 Práctica integradora: ajusta tu red paso a paso

Vas a aplicar el **orden de ajuste** de la sección 8 a otra parte de MNIST: 6,000 imágenes para entrenar, 2,000 para validar y 2,000 de **prueba** (del conjunto oficial de prueba, que no se toca hasta el final). Hazlo en 6 pasos y **no avances sin revisar cada respuesta**.

**Paso 1 ▶️ — Datos y punto de partida.** Entrenamos con los valores por defecto: Adam, `lr=1e-3`, lote 64, 1 capa de 128, 5 épocas.

In [ ]:
mnist_prueba = datasets.MNIST("data", train=False, download=True)
P = dict(X_tr=X_todo[20_000:26_000], y_tr=y_todo[20_000:26_000],
         X_va=X_todo[30_000:32_000], y_va=y_todo[30_000:32_000])
X_prueba = mnist_prueba.data[:2000].float().div(255).view(-1, 784)
y_prueba = mnist_prueba.targets[:2000]

base = entrenar(**P)
print(f"Punto de partida → exactitud validación {base['exactitud_va'][-1]:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

≈**89 %** en validación. Con menos datos (6,000) que en la lección, el punto de partida queda algo más bajo. Esta es la marca a superar.
</details>

**Paso 2 ▶️ — Tasa de aprendizaje con el *LR range test*.**

In [ ]:
tasas, perdidas = buscar_lr(P["X_tr"], P["y_tr"])
LR_SUGERIDA = tasas[perdidas.argmin()] / 10
plt.plot(tasas, perdidas); plt.xscale("log"); plt.axvline(LR_SUGERIDA, color="g", ls="--")
plt.xlabel("tasa (log)"); plt.ylabel("pérdida suavizada"); plt.show()
print(f"Tasa sugerida ≈ {LR_SUGERIDA:.3g}")

<details><summary><b>👀 Ver respuesta</b></summary>

La pérdida toca fondo hacia ≈0.02 y la tasa sugerida queda en ≈**0.002**. Igual que en la lección: es una zona segura, no el número final.
</details>

**Paso 3 ✏️ — Comprueba la tasa con un mini-barrido.** El *LR range test* da una zona segura, no un número exacto. Probamos la sugerencia y un par de valores más altos.

In [ ]:
for lr in [0.002, 0.005, 0.01]:       # ✏️ números redondos alrededor de tu tasa sugerida
    h = entrenar(lr=lr, **P)
    print(f"lr={lr:<6} → exactitud validación {h['exactitud_va'][-1]:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- `0.002` → ≈91 % · `0.005` → ≈91 % · `0.01` → ≈93 %.

Solo con mejorar la tasa ya ganaste **≈4 puntos** sobre el punto de partida (≈89 %). La sugerencia del *LR range test* era prudente; el mini-barrido encontró que se podía subir un poco más.
</details>

**Paso 4 ✏️ — *Scheduler* y capacidad.** Con la mejor tasa, probamos *one-cycle* y una red más ancha.

In [ ]:
LR = 0.01    # ✏️ la mejor tasa del Paso 3
for nombre, sched, neuronas in [("constante", None, 128),
                                ("one-cycle", schedulers["One-cycle"], 128),
                                ("one-cycle", schedulers["One-cycle"], 512)]:
    h = entrenar(lr=LR, scheduler=sched, neuronas=neuronas, **P)
    print(f"{nombre:<10} neuronas={neuronas:<4} → pérdida entrenamiento {h['perdida_tr'][-1]:.3f} · exactitud validación {h['exactitud_va'][-1]:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Constante, 128 neuronas: ≈93 %.
- *One-cycle*, 128 neuronas: ≈93 % también (incluso unas décimas menos). **Aquí el *scheduler* no ayudó**: con solo 5 épocas y estos datos, la diferencia es ruido. En el Ejercicio 5 (8 épocas) sí ganaba 1–2 puntos. Un *scheduler* suele ayudar, pero hay que comprobarlo.
- *One-cycle*, 512 neuronas: ≈94 %. Más capacidad sí aportó ≈1 punto. Su pérdida de entrenamiento ya es muy baja (≈0.02): empieza a memorizar, así que el siguiente paso sería regularizar (Lección 9.8).
</details>

**Paso 5 ▶️ — El examen final, una sola vez.** Entrenamos la configuración elegida y la evaluamos en el conjunto de **prueba**, que no usamos para ninguna decisión.

In [ ]:
NEURONAS = 512    # ✏️ tu elección del Paso 4
final = entrenar(lr=LR, scheduler=schedulers["One-cycle"], neuronas=NEURONAS,
                 X_tr=P["X_tr"], y_tr=P["y_tr"], X_va=X_prueba, y_va=y_prueba)
print(f"Punto de partida (validación) : {base['exactitud_va'][-1]:.1%}")
print(f"Configuración final (PRUEBA)  : {final['exactitud_va'][-1]:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

≈**94 %** en prueba, muy parecido a lo que vimos en validación (≈94 %). Que coincidan es buena señal: no "sobreajustaste tus decisiones" a la validación. Pasaste de ≈89 % a ≈94 %, es decir, **la mitad de los errores**, sin cambiar de modelo ni de datos: solo girando perillas en orden.
</details>

**Paso 6 ✋ — Para pensar.** Mira cuánto mejoraste en total y **qué paso aportó más**. Si tuvieras una hora más de cómputo, ¿qué probarías a continuación?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

El paso que más aportó fue **la tasa de aprendizaje** (≈+4 puntos); la capacidad sumó ≈1 punto y el *scheduler* casi nada en este caso. Es lo que predice el orden de la sección 8.

Con una hora más: más **épocas** con *early stopping* (5 épocas es poco), **más datos** (¡quedan 54,000 imágenes sin usar!), regularización como *dropout* o *weight decay* (Lección 9.8) para la red ancha, y repetir las comparaciones cerradas con otra semilla antes de creer diferencias de décimas. Para imágenes, la gran mejora vendrá de las redes convolucionales (Módulo 10).
</details>

---
## 📚 Para profundizar
- 📘 **Lectura:** Leslie N. Smith, *"Cyclical Learning Rates for Training Neural Networks"* (el *LR range test*) y *"Super-Convergence"* (*one-cycle*).
- 📘 **Guía práctica:** Google Research, *"Deep Learning Tuning Playbook"* (GitHub): cómo ajustan hiperparámetros los equipos profesionales, en orden.
- 📺 **Video:** Andrew Ng, *"Hyperparameter tuning"* y *"Learning rate decay"* (DeepLearning.AI, inglés con subtítulos).
- 🎮 **Interactivo:** TensorFlow Playground (playground.tensorflow.org): cambia tasa, lote, capas y neuronas y mira el efecto en vivo.
- 📘 **Documentación:** PyTorch, `torch.optim.lr_scheduler`.

---
⬅️ **Anterior:** [9.5 Optimizadores: SGD, Momentum, RMSProp y Adam, o cómo bajar la montaña con estilo](9.5-optimizadores.ipynb) · ➡️ **Siguiente:** [9.7 Inicialización de pesos y gradientes que desaparecen o explotan: el teléfono descompuesto](9.7-inicializacion-y-gradientes.ipynb)